# Challenge 1 - EGFR binder generation (Kaggle)Anthropic x Adaptyv Protein Design Competition. Epitope and hotspots come from thePhase 1 cross-species conservation analysis.**Before running:** Settings -> Accelerator -> **GPU P100**, and Settings -> **Internet: On**(internet requires phone verification on your Kaggle account).Pipeline: RFdiffusion (backbones) -> ProteinMPNN soluble (sequences) -> FASTA.Run cells in order. Cell 1 fails fast if the environment is wrong.

## 1. Environment check (run first - fails fast)

In [ ]:
import subprocess, urllib.requestprint(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],                     capture_output=True, text=True).stdout)try:    urllib.request.urlopen("https://pypi.org", timeout=15)    print("internet: OK")except Exception as e:    raise SystemExit("INTERNET IS OFF - enable it in Settings. " + str(e))import torchprint("torch", torch.__version__, "cuda:", torch.cuda.is_available())if not torch.cuda.is_available():    raise SystemExit("No GPU - set Accelerator to GPU P100 in Settings.")

## 2. Build the target constructDownloads PDB 6ARU and trims to EGFR domain I (chain A, PDB residues 1-164) with thecetuximab Fab removed. Nothing to upload.Epitope A = F44, E45, D46, L49, Q52, Y74, D75 (UniProt) = PDB 20, 21, 22, 25, 28, 50, 51.100% identical human/mouse, overlaps the EGF ligand site, and verified to have the samesolvent exposure in this trimmed construct as in the intact receptor.

In [ ]:
import urllib.request, osfrom Bio.PDB import PDBParser, PDBIO, Selectos.makedirs("/kaggle/working/targets", exist_ok=True)raw = "/kaggle/working/6ARU.pdb"if not os.path.exists(raw):    data = urllib.request.urlopen(        "https://files.rcsb.org/download/6ARU.pdb", timeout=120).read()    open(raw, "wb").write(data)class DomainI(Select):    def accept_chain(self, c): return c.id == "A"    def accept_residue(self, r): return r.id[0] == " " and 1 <= r.id[1] <= 164st = PDBParser(QUIET=True).get_structure("x", raw)io = PDBIO(); io.set_structure(st)TARGET = "/kaggle/working/targets/EGFR_A_domainI.pdb"io.save(TARGET, DomainI())n = sum(1 for l in open(TARGET)        if l.startswith("ATOM") and l[12:16].strip() == "CA")print("wrote", TARGET, "with", n, "residues")assert 150 < n < 170, "unexpected residue count"

## 3. Install RFdiffusion (~10 min)

In [ ]:
%%bashset -ecd /kaggle/working[ -d RFdiffusion ] || git clone -q https://github.com/RosettaCommons/RFdiffusion.gitcd RFdiffusion && mkdir -p models && cd modelsfor f in 6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt \         e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt \         f572d396fae9206628714fb2ce00f72e/Complex_beta_ckpt.pt ; do  name=$(basename $f)  [ -f "$name" ] || wget -q "http://files.ipd.uw.edu/pub/RFdiffusion/$f"donels -la

In [ ]:
%%bashset -epip install -q hydra-core omegaconf icecream pyrsistent e3nn opt_einsum_fx 2>&1 | tail -2pip install -q --no-deps dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html 2>&1 | tail -2cd /kaggle/working/RFdiffusion/env/SE3Transformer && pip install -q --no-deps . 2>&1 | tail -2cd /kaggle/working/RFdiffusion && pip install -q --no-deps -e . 2>&1 | tail -2echo INSTALL_DONE

## 4. Run RFdiffusion`contigs=[A1-164/0 55-95]` keeps the target and diffuses a 55-95 aa binder, which keepsevery design inside the competition's minibinder category (40-100 aa). `noise_scale=0`is the standard setting for binder design.Start with `NUM = 20` to confirm the pipeline runs end to end, then raise it. Kagglesessions cap at 12 hours.

In [ ]:
import osos.environ["DGLBACKEND"] = "pytorch"HOTSPOTS = "[A20,A21,A22,A25,A28,A50,A51]"CONTIGS  = "[A1-164/0 55-95]"NUM      = 20      # raise once a first batch succeedsPREFIX   = "/kaggle/working/out/egfr_a"CKPT     = ""      # beta batch: see the next cellos.makedirs("/kaggle/working/out", exist_ok=True)cmd = ("cd /kaggle/working/RFdiffusion && python scripts/run_inference.py"       " inference.output_prefix=" + PREFIX +       " inference.input_pdb=/kaggle/working/targets/EGFR_A_domainI.pdb"       " 'contigmap.contigs=" + CONTIGS + "'"       " 'ppi.hotspot_res=" + HOTSPOTS + "'"       " inference.num_designs=" + str(NUM) +       " denoiser.noise_scale_ca=0 denoiser.noise_scale_frame=0 " + CKPT)print(cmd)get_ipython().system(cmd)

### Optional second batch: the beta checkpointOnly ~15 of 354 binders in Anthropic's campaign had design models with >=20% beta strand,because the default methods overwhelmingly produce all-alpha folds. Track 3 selectionweighs design novelty, so a minority of beta-containing backbones is cheapdifferentiation. Rerun cell 4 with these two values changed.

In [ ]:
# PREFIX = "/kaggle/working/out/egfr_a_beta"# CKPT = ("inference.ckpt_override_path="#         "/kaggle/working/RFdiffusion/models/Complex_beta_ckpt.pt")print("edit cell 4 with the two lines above, then rerun it")

## 5. ProteinMPNN - sequence designDesigns the binder chain (B) only, with target chain A held fixed. Soluble weights, nocysteines.Histidines are deliberately NOT placed here. Switch positions are chosen in Phase 3against specific conserved carboxylates (E45, D46, D75, E97), because the publishedpH-switch work found that random histidine incorporation rarely produces a workingswitch.

In [ ]:
%%bashset -ecd /kaggle/working[ -d ProteinMPNN ] || git clone -q https://github.com/dauparas/ProteinMPNN.gitecho PROTEINMPNN_READY

In [ ]:
import glob, os, subprocesspdbs = sorted(glob.glob("/kaggle/working/out/egfr_a*.pdb"))print(len(pdbs), "backbones from RFdiffusion")os.makedirs("/kaggle/working/seqs", exist_ok=True)for p in pdbs:    cmd = ["python", "/kaggle/working/ProteinMPNN/protein_mpnn_run.py",           "--pdb_path", p, "--pdb_path_chains", "B",           "--out_folder", "/kaggle/working/seqs",           "--num_seq_per_target", "8", "--sampling_temp", "0.1",           "--omit_AAs", "CX", "--use_soluble_model", "--seed", "37"]    r = subprocess.run(cmd, capture_output=True, text=True)    if r.returncode != 0:        print(os.path.basename(p), "FAILED:", r.stderr[-400:])        breakprint("sequence design done")

## 6. Collect into one FASTA and zip for download

In [ ]:
import glob, os, zipfilerecords = []for f in sorted(glob.glob("/kaggle/working/seqs/seqs/*.fa")):    name = os.path.basename(f).replace(".fa", "")    entries, hdr = [], None    for line in open(f):        line = line.strip()        if line.startswith(">"):            hdr = line        elif line:            entries.append((hdr, line))    # entry 0 is the input backbone sequence; the designs follow    for i, (h, s) in enumerate(entries[1:], start=1):        records.append((name + "_s" + str(i), s))out = "/kaggle/working/designs.fasta"with open(out, "w") as fh:    for n, s in records:        fh.write(">" + n + "\n" + s + "\n")print(len(records), "sequences ->", out)if records:    print("example:", records[0][0], records[0][1][:60], "...")with zipfile.ZipFile("/kaggle/working/challenge1_outputs.zip", "w",                     zipfile.ZIP_DEFLATED) as z:    for p in glob.glob("/kaggle/working/out/*.pdb"):        z.write(p, "pdb/" + os.path.basename(p))    if os.path.exists(out):        z.write(out, "designs.fasta")print("zipped -> challenge1_outputs.zip (download from the Output panel)")